In [1]:
import json
from pprint import pprint
import os
import ast
import pandas as pd
import concurrent.futures



In [2]:
# read csv file

df = pd.read_csv('o4mini_annotation_final.csv')
df.head()


,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,t_result,t_corrected_version,t_category,t_notes,h_result,...,llm_errors,num_groups,agreement,text_length,llm_result_final,issues,llm_a1_agreement,llm_a2_agreement,a1_a2_agreement,llm_final_agreement
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",1,NaN,NaN,NaN,1,...,NaN,4.0,TRUE,110,"[\n {\n ""Chest pain"": [\n ""non stop c...",[],TRUE,TRUE,TRUE,TRUE
1,1527339,"cough, nasal congestion, runny nose, cold swea...","['Cold sweat', 'Cough', 'Fatigue', 'Hypogeusia...","""No temporal information found""",No temporal information found,1,NaN,NaN,NaN,1,...,NaN,NaN,TRUE,18,No temporal information found,[],TRUE,TRUE,TRUE,TRUE
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",Symptoms with no original mention should be gr...,NaN,0,...,NaN,2.0,FALSE,13,"[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","['llm formatting issue', 'incorrect grouping',...",FALSE,FALSE,TRUE,FALSE
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",1,NaN,NaN,NaN,0,...,NaN,2.0,0,260,"[\n {\n ""Vaccination site pain"": [\n ...",[],TRUE,0,0,TRUE
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",1,NaN,NaN,NaN,1,...,NaN,2.0,1,118,"[\n {\n ""Syncope"": [\n ""patient faint...",[],1,1,1,1


In [3]:
# get columns from df as new df
data = df[['vaers_id','symptom_text', 'symptom_list','llm_result','llm_result_formatted','final_corrected_version','final_result']]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 6000 non-null   int64 
 1   symptom_text             6000 non-null   object
 2   symptom_list             6000 non-null   object
 3   llm_result               6000 non-null   object
 4   llm_result_formatted     6000 non-null   object
 5   final_corrected_version  896 non-null    object
 6   final_result             6000 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 328.2+ KB


In [4]:
# check if "no temporal information" in final_corrected_version or llm_result_formatted if final_corrected_version is NaN and store in new column "no_temporal_info"
data['no_temporal_info'] = data.apply(lambda row:
    'no temporal information' in str(row['final_corrected_version']).lower() or
    ('final_corrected_version' in str(row['final_corrected_version']).lower() if
    pd.isna(row['final_corrected_version']) else False) or
    'no temporal information' in str(row['llm_result_formatted']).lower(),
    axis=1
)
data['no_temporal_info'].value_counts()
# get rows where no_temporal_info is False as new df
data_with_temporal_info = data[data['no_temporal_info'] == False]
data_with_temporal_info.info()
# we have to remove final_result value 2 from data_with_temporal_info
data_with_temporal_info = data_with_temporal_info[data_with_temporal_info['final_result'] != 2]
data_with_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3794 entries, 0 to 5998
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3794 non-null   int64 
 1   symptom_text             3794 non-null   object
 2   symptom_list             3794 non-null   object
 3   llm_result               3794 non-null   object
 4   llm_result_formatted     3794 non-null   object
 5   final_corrected_version  654 non-null    object
 6   final_result             3794 non-null   int64 
 7   no_temporal_info         3794 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 240.8+ KB
<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   ob

/tmp/ipykernel_605236/3745764098.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['no_temporal_info'] = data.apply(lambda row:


In [5]:
# find data with no temporal info
data_no_temporal_info = data[data['no_temporal_info'] == True]
data_no_temporal_info = data_no_temporal_info[data_no_temporal_info['final_result'] != 2]

data_no_temporal_info.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2191 entries, 1 to 5999
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 2191 non-null   int64 
 1   symptom_text             2191 non-null   object
 2   symptom_list             2191 non-null   object
 3   llm_result               2191 non-null   object
 4   llm_result_formatted     2191 non-null   object
 5   final_corrected_version  238 non-null    object
 6   final_result             2191 non-null   int64 
 7   no_temporal_info         2191 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 139.1+ KB


In [6]:
# take 50 each from data_with_temporal_info and data_no_temporal_info
data_sampled_with_temporal_info = data_with_temporal_info.sample(n=200, random_state=67)
data_sampled_no_temporal_info = data_no_temporal_info.sample(n=100, random_state=67)
# concatenate the two dataframes
data_sampled = pd.concat([data_sampled_with_temporal_info, data_sampled_no_temporal_info])
data_sampled.info()
# save to csv
# data_sampled.to_csv('sample_40.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 300 entries, 133 to 2284
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 300 non-null    int64 
 1   symptom_text             300 non-null    object
 2   symptom_list             300 non-null    object
 3   llm_result               300 non-null    object
 4   llm_result_formatted     300 non-null    object
 5   final_corrected_version  47 non-null     object
 6   final_result             300 non-null    int64 
 7   no_temporal_info         300 non-null    bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 19.0+ KB


In [7]:
# now lets take 40 random samples from data_with_temporal_info with random_state=2
sampled_data = data_with_temporal_info 
sampled_data.info()


<class 'pandas.core.frame.DataFrame'>
Index: 3156 entries, 0 to 5995
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   vaers_id                 3156 non-null   int64 
 1   symptom_text             3156 non-null   object
 2   symptom_list             3156 non-null   object
 3   llm_result               3156 non-null   object
 4   llm_result_formatted     3156 non-null   object
 5   final_corrected_version  651 non-null    object
 6   final_result             3156 non-null   int64 
 7   no_temporal_info         3156 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 200.3+ KB


In [8]:
# eg = data_with_temporal_info[['vaers_id','symptom_text', 'symptom_list', 'llm_result_formatted','final_corrected_version','final_result']]
# eg.to_csv('openai_gold_with_temporal.csv', index=False)

In [9]:
# assume temporal information exists for all
# 0 shot prompt template
example = """
[
  {"Erythema": ["redness in neck"]},
  {
    "Pain in extremity": ["sore arm"],
    "Pruritus": ["itchy feeling"]
  },
  {"Swelling": ["mild arm swelling"]}
]
"""

input_template_mention = """
Ignore previous conversations.

Clinical Notes: {text}

Task:
    - Use the adverse effect list below:
     {suggest}
   - Reorder the list based on the sequence implied in the clinical notes.
   - If several adverse effects are mentioned **together without a clear temporal order**, group them into the same block (inside a single dictionary).
   - Do **not skip any terms** from the list — include all terms in the output.
   - For each term, extract the closest matching phrase from the clinical notes; if a term is not mentioned, assign "none" as its value.

Important Rules:
- Mention each symptom **only once** — at the first time it appears or becomes relevant.
- Do **NOT** treat different sentences as different times unless there is an explicit temporal indicator (e.g., "then", "after that", "later", specific dates or times).
- If multiple symptoms are mentioned across multiple sentences **without clear timeline separation**, **group them together**.
- Only separate symptoms into different blocks when the clinical notes clearly describe a **sequence or timing** between them.

Finally, return your answer **only** in valid JSON format —
using a **list of dictionaries** to represent temporal progression and grouping, as shown below:
{example}
"""

In [ ]:
import time
import random
from anthropic import Anthropic
import anthropic

# =========================
# Claude model + API key (change in-code)
# =========================
MODEL_ID = ""   # e.g., "claude-haiku-4-5","claude-sonnet-4-0","claude-3-7-sonnet-latest"
MAX_TOKENS = 512
TEMPERATURE = 0

# Put your Anthropic key here (or set env ANTHROPIC_API_KEY)
ANTHROPIC_API_KEY = "" 

client = Anthropic(
    api_key=ANTHROPIC_API_KEY if ANTHROPIC_API_KEY else None,
    max_retries=5,
)

def set_model(model_id: str):
    global MODEL_ID
    MODEL_ID = model_id

def _extract_text(msg) -> str:
    parts = []
    for block in getattr(msg, "content", []) or []:
        btype = getattr(block, "type", None) if not isinstance(block, dict) else block.get("type")
        if btype == "text":
            txt = getattr(block, "text", None) if not isinstance(block, dict) else block.get("text")
            if txt:
                parts.append(txt)
    return "".join(parts).strip()

def llm(prompt: str, retries: int = 6, delay: float = 1.0, backoff: float = 2.0) -> str:
    for attempt in range(retries):
        try:
            msg = client.messages.create(
                model=MODEL_ID,
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
                messages=[{"role": "user", "content": prompt}],
            )
            return _extract_text(msg)
        except (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.APIStatusError) as e:
            time.sleep(delay + random.random())
            delay *= backoff
        except Exception as e:
            return f"__ERROR__:CLAUDE:{type(e).__name__}:{e}"
    return "__ERROR__:CLAUDE:MaxRetriesExceeded"


def get_ordered_symptoms(symptom_list):
    symptoms = []
    if isinstance(symptom_list, list):
        for sym_dict in symptom_list:
            symptoms.append(list(sym_dict.keys()))
    return symptoms

def call_llm(row):
    prompt = input_template_mention.format(
        text=row['symptom_text'],  # Truncate for speed
        suggest=row['symptom_list'],
        # few_shot_examples=few_shot_examples,
        example=example
    )
    return llm(prompt)


In [11]:

batch_size = 500
for i in range(0, len(sampled_data), batch_size):
    batch = sampled_data.iloc[i:i+batch_size]
    print(f"Processing records {i} to {i + batch_size}")
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(call_llm, [row for _, row in batch.iterrows()]))

    sampled_data.loc[batch.index, 'claude_result'] = results
    sampled_data.to_csv(f"claude_baseline_with_temporal_part_{i}.csv", index=False)  # Save checkpoint

   


Processing records 0 to 500
Processing records 500 to 1000
Processing records 1000 to 1500
Processing records 1500 to 2000
Processing records 2000 to 2500
Processing records 2500 to 3000
Processing records 3000 to 3500


In [12]:
import re
def parse_json_best_effort(text: str):
    if not text:
        return None
    try:
        return json.loads(text)
    except Exception:
        pass
    m = re.search(r"```(?:json)?\s*(\{.*?\}|\[.*?\])\s*```", text, flags=re.S)
    if m:
        try:
            return json.loads(m.group(1))
        except Exception:
            pass
    start = min([p for p in (text.find("{"), text.find("[")) if p != -1], default=-1)
    if start != -1:
        for end in range(len(text) - 1, start, -1):
            if text[end] in "}]":
                chunk = text[start:end + 1]
                try:
                    return json.loads(chunk)
                except Exception:
                    continue
    return None

In [13]:
res_lst = []
for row in sampled_data.iterrows():
    res = parse_json_best_effort(row[1]['claude_result'])
    res_lst.append(res)

# store res_lst in reslts_df
sampled_data['final_result_cleaned'] = res_lst
sampled_data.head()



,vaers_id,symptom_text,symptom_list,llm_result,llm_result_formatted,final_corrected_version,final_result,no_temporal_info,claude_result,final_result_cleaned
0,1420769,"About 4-5 days after my second dose, I began h...","['Chest discomfort', 'Chest pain', 'Electrocar...","[ \n { \n ""Chest pain"": [""non stop chest...","[\n {\n ""Chest pain"": [\n ""non stop c...",NaN,1,False,"```json\n[\n {\n ""Chest discomfort"": [""non...",[{'Chest discomfort': ['non stop chest pain th...
2,1831772,"Muscle pain, headache, nausea. Muscle pain ins...","['Headache', 'Immediate post-injection reactio...","[\n {\n ""Headache"": [""none""],\n ""Immedi...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...","[\n {\n ""Myalgia"": [\n ""Muscle pain""\...",0,False,"```json\n[\n {\n ""Myalgia"": [""Muscle pain ...","[{'Myalgia': ['Muscle pain instantly'], 'Immed..."
3,1021935,back pain/soreness in his lower back/had bad l...,"['Arthralgia', 'Back pain', 'Nausea', 'Pain in...","[\n {\n ""Vaccination site pain"": [""injecti...","[\n {\n ""Vaccination site pain"": [\n ...",NaN,1,False,"```json\n[\n {\n ""Vaccination site pain"": ...",[{'Vaccination site pain': ['injection site pa...
4,2749020,The above vaccines administered without incide...,"['Dizziness', 'Fall', 'Fatigue', 'Head injury'...","[\n {\n ""Syncope"": [""patient fainted""],\n ...","[\n {\n ""Syncope"": [\n ""patient faint...",NaN,1,False,"```json\n[\n {\n ""Syncope"": [""patient fain...","[{'Syncope': ['patient fainted'], 'Fall': ['th..."
5,940509,"about 4 hrs after vaccination ee had ""tunnel ...","['Dizziness', 'Faeces soft', 'Fatigue', 'Flatu...","[\n {\n ""Tunnel vision"": [""tunnel vision""]...","[\n {\n ""Tunnel vision"": [\n ""tunnel ...",NaN,1,False,"```json\n[\n {\n ""Tunnel vision"": [""tunnel...","[{'Tunnel vision': ['tunnel vision'], 'Dizzine..."


In [ ]:
sampled_data.to_csv(f"claude_baseline_0shot_with_temporal_final.csv", index=False)  # Save checkpoint
